In [13]:
import os
import pandas as pd
import numpy as np
from sklearn.feature_selection import (
    SelectKBest, chi2, f_classif, mutual_info_classif, VarianceThreshold
)

# Rutas
INPUT_PATH = r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\featuring"
OUTPUT_PATH = r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\selected"

# Configuraciones
normalizations = ["Standard", "MinMax", "MaxAbs", "Robust", "NoNorm"]
types = ["ORIGINAL", "FE"]
methods = ["CHI2", "ANOVA", "MI", "VAR", "ALL"]

def guardar_csv(df, nombre_archivo):
    path_csv = os.path.join(OUTPUT_PATH, nombre_archivo)
    df.to_csv(path_csv, index=False)
    print(f"📥 Métricas guardadas: {path_csv}")

def seleccionar_variables(X_train, y_train, method, k=100):
    if method == "CHI2":
        if (X_train < 0).any().any():
            print("⚠️ Saltando CHI2 por contener negativos")
            return None, None
        selector = SelectKBest(score_func=chi2, k=k)
    elif method == "ANOVA":
        selector = SelectKBest(score_func=f_classif, k=k)
    elif method == "MI":
        selector = SelectKBest(score_func=mutual_info_classif, k=k)
    elif method == "VAR":
        selector = VarianceThreshold()
        selector.fit(X_train)
        support = selector.get_support()
        return X_train.columns[support], selector.variances_[support]
    elif method == "ALL":
        return X_train.columns, np.ones(X_train.shape[1])
    else:
        raise ValueError(f"Método desconocido: {method}")

    selector.fit(X_train, y_train)
    support = selector.get_support()
    columnas = X_train.columns[support]
    try:
        scores = selector.scores_[support]
    except:
        scores = np.full(len(columnas), np.nan)
    return columnas, scores

# Loop principal
for norm in normalizations:
    for tipo in types:
        nombre_combo = f"{norm} - {tipo}"
        print(f"\n🔍 Procesando {nombre_combo}...")

        try:
            X_file = os.path.join(INPUT_PATH, norm, f"X_train_{norm}_{tipo}.parquet")
            y_file = os.path.join(INPUT_PATH, norm, f"y_train_{norm}_{tipo}.parquet")
            X_train = pd.read_parquet(X_file)
            y_train = pd.read_parquet(y_file).squeeze()
        except Exception as e:
            print(f"❌ Error cargando datos para {nombre_combo}: {e}")
            continue

        for method in methods:
            nombre_salida = f"{norm}_{tipo}_{method}"
            try:
                columnas, scores = seleccionar_variables(X_train, y_train, method)
                if columnas is None or scores is None:
                    continue

                if len(columnas) != len(scores):
                    raise ValueError(f"Longitudes distintas: columnas={len(columnas)}, scores={len(scores)}")

                df_metricas = pd.DataFrame({
                    "variable": columnas,
                    "score": scores
                }).sort_values(by="score", ascending=False)

                guardar_csv(df_metricas, f"metricas_{nombre_salida}.csv")

                df_filtrado = X_train[columnas]
                df_filtrado.to_parquet(os.path.join(OUTPUT_PATH, f"X_train_{nombre_salida}.parquet"), index=False)
                y_train.to_frame(name='nivel_triage').to_parquet(
                    os.path.join(OUTPUT_PATH, f"y_train_{nombre_salida}.parquet"), index=False)

                print(f"✔ {nombre_combo} - {method}: {df_filtrado.shape[1]} variables seleccionadas")
            except Exception as e:
                print(f"❌ Error al guardar {method} en {nombre_combo}: {e}")



🔍 Procesando Standard - ORIGINAL...
⚠️ Saltando CHI2 por contener negativos
📥 Métricas guardadas: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\selected\metricas_Standard_ORIGINAL_ANOVA.csv
✔ Standard - ORIGINAL - ANOVA: 100 variables seleccionadas
📥 Métricas guardadas: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\selected\metricas_Standard_ORIGINAL_MI.csv
✔ Standard - ORIGINAL - MI: 100 variables seleccionadas
📥 Métricas guardadas: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\selected\metricas_Standard_ORIGINAL_VAR.csv
✔ Standard - ORIGINAL - VAR: 539 variables seleccionadas
📥 Métricas guardadas: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\selected\metricas_Standard_ORIGINAL_ALL.csv
✔ Standard - ORIGINAL - ALL: 539 variables seleccionadas

🔍 Procesando Standard - FE...
⚠️ Saltando CHI2 por contener negativos
📥 Métricas guardadas: C:\Users\Gonzalo\Documents\GitHub\Te